In [ ]:
# =============================================================================
# FUNCTION CALLING & MCP
# =============================================================================
#
# ---------------------------------------------------------------------------
# WHERE ARE WE ON THE PATH?
# ---------------------------------------------------------------------------
#   Pretrain → SFT → DPO → Quantization  = teach + shrink the model
#   THIS NOTEBOOK                        = give the model HANDS
#
# Easy analogy:
#   A language model alone is a very smart brain in a locked room.
#   It can talk about prices, but it cannot LOOK UP a live price.
#   Function calling = give it a phone to call your code.
#   MCP (Model Context Protocol) = a standard "phone book + dial tone"
#   so many apps can expose tools the same way (more on that later).
#
# Why care for your earpiece product?
#   On a live sales call you need real facts: plan price, CRM discount,
#   calendar, docs — not guesses. The model decides WHEN to call a tool;
#   YOUR Python code does the real lookup and returns the answer.
#
#
# ---------------------------------------------------------------------------
# THE BIG PICTURE (agent loop)
# ---------------------------------------------------------------------------
#   1) User asks something  ("What's the starter plan price?")
#   2) Prompt includes TOOL DESCRIPTIONS (this cell builds those tools)
#   3) Model either answers in plain text OR outputs a JSON function_call
#   4) Your code parses JSON → runs the real Python function
#   5) You feed the tool RESULT back to the model → final spoken answer
#
# This cell = step 2's ingredients: real functions + a schema the model reads.
#
#
# ---------------------------------------------------------------------------
# PART A — mock "backend" (fake databases)
# ---------------------------------------------------------------------------
# In production these would be API/DB calls. Here they are dicts so we can
# learn the pattern without networking.

PRODUCT_DB = {
    "starter": 99,
    "professional": 299,
    "enterprise": "custom (contact sales)",
}

CUSTOMER_DB = {
    "acme_corp": {"tier": "enterprise", "discount": 0.15},
    "startup_inc": {"tier": "starter", "discount": 0.0},
}


# ---------------------------------------------------------------------------
# PART B — the actual Python functions the model may call
# ---------------------------------------------------------------------------
# These are normal functions. The model never "runs" them by magic.
# YOU run them after the model asks for a call by name + arguments.

def get_product_price(product_name: str) -> str:
    """Return the price of a product plan (monthly)."""
    product_name = product_name.lower()
    if product_name in PRODUCT_DB:
        price = PRODUCT_DB[product_name]
        return f"The {product_name} plan costs ${price} per month."
    return "Product not found."


def get_customer_discount(company: str) -> str:
    """Return the discount and tier for a customer company."""
    company = company.lower()
    if company in CUSTOMER_DB:
        info = CUSTOMER_DB[company]
        pct = info["discount"] * 100
        return f"{company} is on the {info['tier']} tier with a {pct}% discount."
    return "Company not found."


# Map name → callable. Later we dispatch: TOOL_IMPL[name](**arguments)
TOOL_IMPL = {
    "get_product_price": get_product_price,
    "get_customer_discount": get_customer_discount,
}


# ---------------------------------------------------------------------------
# PART C — tool REGISTRY (JSON Schema the model sees)
# ---------------------------------------------------------------------------
# Critical idea: the model does NOT see your Python source.
# It only sees this list: name + description + argument shapes.
# That is how it knows WHAT it can call and WHICH args to fill in.
#
# Shape matches common "tools" / OpenAI-style function schemas:
#   name        → which function
#   description → when to use it (the model reads this carefully)
#   parameters  → JSON Schema for the arguments object

tools = [
    {
        "name": "get_product_price",
        "description": "Get the monthly price of a product plan.",
        "parameters": {
            "type": "object",
            "properties": {
                "product_name": {
                    "type": "string",
                    "description": "Plan name (starter, professional, enterprise)",
                }
            },
            "required": ["product_name"],
        },
    },
    {
        "name": "get_customer_discount",
        "description": "Get the discount and tier for a customer company.",
        "parameters": {
            "type": "object",
            "properties": {
                "company": {
                    "type": "string",
                    "description": "Company name (acme_corp, startup_inc)",
                }
            },
            "required": ["company"],
        },
    },
]

print("Tools ready:", [t["name"] for t in tools])
print(get_product_price("starter"))
print(get_customer_discount("acme_corp"))


In [ ]:
# =============================================================================
# PROMPT THE MODEL TO USE TOOLS
# =============================================================================
#
# ---------------------------------------------------------------------------
# WHAT THIS CELL FIXES / TEACHES
# ---------------------------------------------------------------------------
# Previous version put a markdown fence (```json ... ```) INSIDE a Python
# triple-quoted f-string. That compiles, but in Jupyter/Cursor the nested
# ``` often breaks the editor view — the cell looks cut off mid-string.
#
# Fix: describe the JSON shape in plain text (no nested fences).
# Also remember: in an f-string, literal braces must be doubled:
#   {{"name": "x"}}  →  model sees  {"name": "x"}
# Single {ground_truth} is an f-string hole (filled with the variable).
#
# Flow this cell enables:
#   tools list  →  pasted into the prompt  →  model can request a call
#

import json


def build_prompt_with_tools(ground_truth, user_utterance, tools):
    """Build an instruction prompt that lists tools and the call format.

    ground_truth   : optional context (call notes, product facts, etc.)
    user_utterance : what the human just said
    tools          : the registry list from the previous cell
    """
    # Pretty-print the tool schemas so the model can read names + args.
    tool_desc = json.dumps(tools, indent=2)

    # Example call shown to the model (doubled braces = literal JSON braces).
    example_call = """
{
  "function_call": {
    "name": "<function_name>",
    "arguments": {"arg_name": "value"}
  }
}
""".strip()

    prompt = f"""You are an AI sales coach. You can call external functions to retrieve real-time data.

Available tools (JSON):
{tool_desc}

If you need a tool, reply with ONLY a JSON object in this exact shape:
{example_call}

Rules:
- Use a tool when you need a live fact (price, discount, etc.).
- If you do not need a tool, reply with a normal helpful answer (plain text, no JSON).
- Never invent tool results — only request a call; the system will run it.

Context / ground truth: {ground_truth}
User: {user_utterance}
Assistant:"""
    return prompt


# Demo: build a prompt for a question that SHOULD trigger a tool.
demo_prompt = build_prompt_with_tools(
    ground_truth="Live sales call. Customer asked about pricing.",
    user_utterance="What's the monthly price of the starter plan?",
    tools=tools,
)
print(demo_prompt)
print("\n--- prompt length:", len(demo_prompt), "chars ---")


In [ ]:
# =============================================================================
# PARSE + EXECUTE (the middle of the agent loop)
# =============================================================================
#
# The model does not execute tools. YOUR code does:
#   model text  →  find JSON  →  read name + arguments  →  TOOL_IMPL[name](**args)
#
# Below we fake a model reply (as if the LLM already chose a tool),
# then run the same path you would after a real generate() call.
#

import re


def extract_function_call(model_text: str):
    """Return {"name": ..., "arguments": {...}} or None if plain text."""
    # Find the first {...} JSON object in the reply (models sometimes add chatter).
    match = re.search(r"\{[\s\S]*\}", model_text)
    if not match:
        return None
    try:
        payload = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

    # Support either {"function_call": {...}} or a bare {"name", "arguments"}.
    call = payload.get("function_call", payload)
    if not isinstance(call, dict) or "name" not in call:
        return None
    args = call.get("arguments", {})
    if not isinstance(args, dict):
        return None
    return {"name": call["name"], "arguments": args}


def run_tool(call: dict) -> str:
    """Dispatch one parsed call through TOOL_IMPL."""
    name = call["name"]
    args = call["arguments"]
    if name not in TOOL_IMPL:
        return f"Error: unknown tool '{name}'."
    try:
        return TOOL_IMPL[name](**args)
    except TypeError as e:
        return f"Error: bad arguments for '{name}': {e}"


def handle_model_reply(model_text: str) -> str:
    """If the model requested a tool, run it; else return the text as-is."""
    call = extract_function_call(model_text)
    if call is None:
        return model_text.strip()
    print("Model requested tool:", call)
    result = run_tool(call)
    print("Tool result:", result)
    # In a full agent loop you would send `result` back into another prompt
    # so the model can phrase a natural final answer for the earpiece.
    return result


# --- Simulated model outputs (pretend generate() returned these) ---
fake_tool_reply = """
{
  "function_call": {
    "name": "get_product_price",
    "arguments": {"product_name": "starter"}
  }
}
"""

fake_plain_reply = "Sounds good — happy to walk through pricing whenever you're ready."

print("=== case: model wants a tool ===")
print("Final:", handle_model_reply(fake_tool_reply))
print()
print("=== case: model answers directly ===")
print("Final:", handle_model_reply(fake_plain_reply))


In [ ]:
# =============================================================================
# HOW THIS CONNECTS TO MCP (Model Context Protocol)
# =============================================================================
#
# What you built above is "DIY function calling":
#   - You wrote tool schemas by hand
#   - You stuffed them into a prompt
#   - You parsed JSON and called Python yourself
#
# That pattern is the CORE idea behind every tool-using agent.
#
# MCP standardizes the same idea across apps:
#
#   ┌─────────────┐         ┌──────────────────┐         ┌─────────────────┐
#   │ Host / IDE  │ ◄─────► │ MCP Client        │ ◄─────► │ MCP Server      │
#   │ (Cursor)    │         │ (talks protocol)  │         │ (exposes tools) │
#   └─────────────┘         └──────────────────┘         └─────────────────┘
#
# Easy mapping to THIS notebook:
#   tools list     ≈  what an MCP server advertises via tools/list
#   function_call  ≈  tools/call  { name, arguments }
#   TOOL_IMPL[...] ≈  the server's real handler (DB, API, file, …)
#   tool result    ≈  tools/call response content sent back to the model
#
# Why MCP exists (one sentence):
#   So every editor / agent does not invent a private tool format —
#   servers expose tools once; many clients can use them.
#
# For your earpiece copilot later:
#   CRM lookup, calendar, playbook search can each be an MCP server (or
#   plain functions first). Start with the DIY loop above; swap the
#   transport for MCP when you want reusable, shareable tools.
#
# Mental model:
#   Function calling = the skill (model asks → code runs → result returns)
#   MCP              = a shared socket + catalog for that skill
#

print("DIY function calling: DONE (schemas + prompt + parse + execute)")
print("MCP: same loop, standardized client/server transport")
print()
print("Next practice: change user_utterance to ask about acme_corp discount,")
print("fake a get_customer_discount call, and run handle_model_reply(...).")
